In [1]:
# W7D4 - CIA Vanna Integration

from pathlib import Path
import sys

print("=" * 60)
print("W7D4 - CIA Vanna Integration")
print("=" * 60)

print("Python Version:", sys.version)
print("Working Directory:", Path.cwd())
print("Project Setup: Started")

W7D4 - CIA Vanna Integration
Python Version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Working Directory: c:\Users\Prave\OneDrive\Desktop\SQL-Day1\SQL-Day3\Pandas-Advanced\Week-3-Day-5\Week-4\Week-4-Day-5-Capstone\Week-5-Day-1-PandasAI\Week-5-Day-3-DuckDB\Week-6\W6D5-NL-SQL-Dashboard\week_7\W7D4-CIA-Vanna-Integration
Project Setup: Started


In [2]:
import importlib.metadata as metadata

packages = ["vanna", "fastapi", "uvicorn", "pydantic", "ollama"]

for package in packages:
    try:
        version = metadata.version(package)
        print(f"{package}: {version}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

vanna: 2.0.2
fastapi: 0.162.4
uvicorn: 0.54.0
pydantic: 2.14.2
ollama: 0.6.3


In [3]:
import ollama

try:
    response = ollama.list()

    print("Ollama Connection: SUCCESS")
    print("\nAvailable Models:")

    for model in response.models:
        print("-", model.model)

except Exception as e:
    print("Ollama Connection: FAILED")
    print("Error:", e)

Ollama Connection: SUCCESS

Available Models:
- llama3.2:3b
- qwen2.5:3b


In [4]:
import sqlite3
from pathlib import Path

# Existing CRM database location
db_path = Path.cwd().parent.parent / "crm_database.db"

print("Database Path:", db_path)
print("Database Exists:", db_path.exists())

if db_path.exists():
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()

        cursor.execute(
            "SELECT name FROM sqlite_master WHERE type='table'"
        )
        tables = cursor.fetchall()

        print("Available Tables:", tables)

        cursor.execute("SELECT COUNT(*) FROM customers")
        total_records = cursor.fetchone()[0]

        print("Total CRM Records:", total_records)
else:
    print("ERROR: CRM database not found!")

Database Path: c:\Users\Prave\OneDrive\Desktop\SQL-Day1\SQL-Day3\Pandas-Advanced\Week-3-Day-5\Week-4\Week-4-Day-5-Capstone\Week-5-Day-1-PandasAI\Week-5-Day-3-DuckDB\Week-6\W6D5-NL-SQL-Dashboard\crm_database.db
Database Exists: True
Available Tables: [('customers',)]
Total CRM Records: 100


In [5]:
import sqlite3

with sqlite3.connect(db_path) as conn:
    cursor = conn.cursor()

    queries = {
        "Total Customers": "SELECT COUNT(*) FROM customers",
        "Total Deal Value": "SELECT SUM(deal_value) FROM customers",
        "Won Deals": "SELECT COUNT(*) FROM customers WHERE deal_stage = 'Won'",
        "Lost Deals": "SELECT COUNT(*) FROM customers WHERE deal_stage = 'Lost'"
    }

    for title, query in queries.items():
        cursor.execute(query)
        result = cursor.fetchone()[0]
        print(f"{title}: {result}")

Total Customers: 100
Total Deal Value: 24875000
Won Deals: 22
Lost Deals: 10


In [6]:
import inspect
from vanna import Agent, AgentConfig
from vanna.servers.fastapi import VannaFastAPIServer

print("Agent:", inspect.signature(Agent))
print("\nAgentConfig:", inspect.signature(AgentConfig))
print("\nFastAPI Server:", inspect.signature(VannaFastAPIServer))

Agent: (llm_service: vanna.core.llm.base.LlmService, tool_registry: vanna.core.registry.ToolRegistry, user_resolver: vanna.core.user.resolver.UserResolver, agent_memory: vanna.capabilities.agent_memory.base.AgentMemory, conversation_store: Optional[vanna.core.storage.base.ConversationStore] = None, config: vanna.core.agent.config.AgentConfig = AgentConfig(max_tool_iterations=10, stream_responses=True, auto_save_conversations=True, include_thinking_indicators=True, temperature=0.7, max_tokens=None, ui_features=UiFeatures(feature_group_access={<UiFeature.UI_FEATURE_SHOW_TOOL_NAMES: 'tool_names'>: ['admin', 'user'], <UiFeature.UI_FEATURE_SHOW_TOOL_ARGUMENTS: 'tool_arguments'>: ['admin'], <UiFeature.UI_FEATURE_SHOW_TOOL_ERROR: 'tool_error'>: ['admin'], <UiFeature.UI_FEATURE_SHOW_TOOL_INVOCATION_MESSAGE_IN_CHAT: 'tool_invocation_message_in_chat'>: ['admin'], <UiFeature.UI_FEATURE_SHOW_MEMORY_DETAILED_RESULTS: 'memory_detailed_results'>: ['admin']}), audit_config=AuditConfig(enabled=True, lo

In [7]:
from vanna import AgentConfig
from vanna.integrations.local.agent_memory.in_memory import DemoAgentMemory
from vanna.integrations.local import MemoryConversationStore
from vanna.core.registry import ToolRegistry
from vanna.core.user.resolver import UserResolver
from vanna.core.user.models import User
from vanna.core.user.request_context import RequestContext

# Tool Registry
tool_registry = ToolRegistry()

# Agent Memory
agent_memory = DemoAgentMemory(max_items=1000)

# Conversation Store
conversation_store = MemoryConversationStore()

# User Resolver
class LocalUserResolver(UserResolver):
    async def resolve_user(self, request_context: RequestContext) -> User:
        return User(
            id="crm_user_001",
            username="CRM Analyst",
            group_memberships=["user"]
        )

user_resolver = LocalUserResolver()

# Agent Configuration
agent_config = AgentConfig(
    max_tool_iterations=5,
    stream_responses=False,
    temperature=0.0
)

print("Tool Registry: Initialized")
print("Agent Memory: Initialized")
print("Conversation Store: Initialized")
print("User Resolver: Initialized")
print("Agent Configuration: Initialized")

Tool Registry: Initialized
Agent Memory: Initialized
Conversation Store: Initialized
User Resolver: Initialized
Agent Configuration: Initialized


In [8]:
from vanna.integrations.ollama import OllamaLlmService

ollama_llm = OllamaLlmService(
    model="qwen2.5:3b",
    host="http://localhost:11434",
    timeout=240.0,
    num_ctx=8192,
    temperature=0.1
)

print("Ollama LLM initialized successfully!")
print("Model: qwen2.5:3b")
print("Host: http://localhost:11434")

Ollama LLM initialized successfully!
Model: qwen2.5:3b
Host: http://localhost:11434


In [9]:
from vanna import Tool
from vanna.core.tool.models import ToolResult
from vanna.capabilities.sql_runner.models import RunSqlToolArgs
from vanna.integrations.sqlite import SqliteRunner

# SQLite SQL Runner
sql_runner = SqliteRunner(database_path=str(db_path))

# Custom CRM SQL Tool
class CRMQueryTool(Tool):

    @property
    def name(self) -> str:
        return "run_sql"

    @property
    def description(self) -> str:
        return "Execute read-only SQL queries on the CRM customers database."

    def get_args_schema(self):
        return RunSqlToolArgs

    async def execute(self, context, args: RunSqlToolArgs) -> ToolResult:
        try:
            result = await sql_runner.run_sql(args, context)

            return ToolResult(
                success=True,
                result_for_llm=result.to_string(index=False)
            )

        except Exception as e:
            return ToolResult(
                success=False,
                result_for_llm=f"SQL execution failed: {e}",
                error=str(e)
            )

crm_sql_tool = CRMQueryTool()

print("CRM SQL Tool initialized successfully!")
print("Tool Name:", crm_sql_tool.name)
print("Tool Description:", crm_sql_tool.description)

CRM SQL Tool initialized successfully!
Tool Name: run_sql
Tool Description: Execute read-only SQL queries on the CRM customers database.


In [10]:
tool_registry.register_local_tool(
    crm_sql_tool,
    access_groups=["admin", "user"]
)

print("CRM SQL Tool registered successfully!")
print("Registered Tool: run_sql")
print("Access Groups: admin, user")

CRM SQL Tool registered successfully!
Registered Tool: run_sql
Access Groups: admin, user


In [11]:
crm_system_prompt = """
You are an expert CRM Data Analyst and SQL Assistant.

DATABASE:
- SQLite database
- Table name: customers

TABLE SCHEMA:
1. customer_id TEXT
2. customer_name TEXT
3. company TEXT
4. lead_source TEXT
5. sales_rep TEXT
6. deal_value INTEGER
7. deal_stage TEXT
8. status TEXT
9. created_date TIMESTAMP
10. last_activity TIMESTAMP

INSTRUCTIONS:
- Answer CRM questions using SQL queries.
- Use the available run_sql tool to execute SQL.
- Generate valid SQLite SQL syntax.
- Use only the customers table and its listed columns.
- For customer counts, use COUNT(*).
- For deal value totals, use SUM(deal_value).
- Display monetary values in Indian Rupees (INR).
- Do not modify or delete database records.
- Never invent query results.
- Summarize the actual SQL results clearly.
"""

from vanna.core.system_prompt.default import DefaultSystemPromptBuilder

crm_prompt_builder = DefaultSystemPromptBuilder(
    base_prompt=crm_system_prompt
)

print("CRM System Prompt configured successfully!")
print("Database: SQLite")
print("Table: customers")
print("SQL Tool: run_sql")

CRM System Prompt configured successfully!
Database: SQLite
Table: customers
SQL Tool: run_sql


In [12]:
from vanna import Agent

agent = Agent(
    llm_service=ollama_llm,
    tool_registry=tool_registry,
    user_resolver=user_resolver,
    agent_memory=agent_memory,
    conversation_store=conversation_store,
    config=agent_config,
    system_prompt_builder=crm_prompt_builder
)

print("Vanna Agent initialized successfully!")
print("LLM: qwen2.5:3b")
print("Database: CRM SQLite")
print("Tool: run_sql")
print("Agent Status: READY")

Vanna Agent initialized successfully!
LLM: qwen2.5:3b
Database: CRM SQLite
Tool: run_sql
Agent Status: READY


In [13]:
from vanna.servers.fastapi import VannaFastAPIServer

vanna_server = VannaFastAPIServer(agent=agent)

app = vanna_server.create_app()

print("Vanna FastAPI Server initialized successfully!")
print("FastAPI Application:", app.title)
print("Server Status: READY")

Vanna FastAPI Server initialized successfully!
FastAPI Application: Vanna Agents API
Server Status: READY


In [14]:
from uuid import uuid4
from typing import Optional
from fastapi import HTTPException
from pydantic import BaseModel
from vanna.servers.base.models import ChatRequest

# API Request Model
class SQLAnalystRequest(BaseModel):
    question: str
    conversation_id: Optional[str] = None


# CIA SQL Analyst Endpoint
@app.post("/cia/sql-analyst")
async def sql_analyst(request: SQLAnalystRequest):

    try:
        chat_request = ChatRequest(
            message=request.question,
            conversation_id=request.conversation_id or str(uuid4()),
            metadata={}
        )

        chat_response = await vanna_server.chat_handler.handle_poll(
            chat_request
        )

        answers = []

        for chunk in chat_response.chunks:
            if chunk.simple:
                answers.append(chunk.simple)

        return {
            "success": True,
            "conversation_id": chat_response.conversation_id,
            "request_id": chat_response.request_id,
            "answer": answers
        }

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=str(e)
        )


print("CIA SQL Analyst endpoint registered!")
print("Method: POST")
print("Endpoint: /cia/sql-analyst")

CIA SQL Analyst endpoint registered!
Method: POST
Endpoint: /cia/sql-analyst


In [15]:
print("Registered API Routes:")
print("-" * 50)

for route in app.routes:
    if hasattr(route, "methods"):
        print(
            "Path:", route.path,
            "| Methods:", route.methods
        )

# Verify CIA endpoint
endpoint_exists = any(
    route.path == "/cia/sql-analyst"
    and "POST" in getattr(route, "methods", set())
    for route in app.routes
)

print("\nCIA SQL Analyst Endpoint:", endpoint_exists)

Registered API Routes:
--------------------------------------------------
Path: /openapi.json | Methods: {'HEAD', 'GET'}
Path: /docs | Methods: {'HEAD', 'GET'}
Path: /docs/oauth2-redirect | Methods: {'HEAD', 'GET'}
Path: /redoc | Methods: {'HEAD', 'GET'}
Path: / | Methods: {'GET'}
Path: /api/vanna/v2/chat_sse | Methods: {'POST'}
Path: /api/vanna/v2/chat_poll | Methods: {'POST'}
Path: /health | Methods: {'GET'}
Path: /cia/sql-analyst | Methods: {'POST'}

CIA SQL Analyst Endpoint: True


In [16]:
import asyncio

test_request = SQLAnalystRequest(
    question="How many customers are there in the CRM database?"
)

response = await sql_analyst(test_request)

print("API Test Result")
print("=" * 50)
print("Success:", response["success"])
print("Conversation ID:", response["conversation_id"])
print("Answer:", response["answer"])

API Test Result
Success: True
Conversation ID: 265307f6-6b0f-48ba-a52f-cf6b76d59562
Answer: [{'type': <SimpleComponentType.TEXT: 'text'>, 'semantic_type': None, 'metadata': None, 'text': 'There are 100 customers in the CRM database.'}]


In [17]:
test_request_2 = SQLAnalystRequest(
    question="What is the total deal value of all customers?"
)

response_2 = await sql_analyst(test_request_2)

print("API Test Result")
print("=" * 50)
print("Success:", response_2["success"])
print("Answer:", response_2["answer"])

API Test Result
Success: True
Answer: [{'type': <SimpleComponentType.TEXT: 'text'>, 'semantic_type': None, 'metadata': None, 'text': 'The total deal value of all customers is ₹24,87,5000.'}]


In [18]:
test_request_3 = SQLAnalystRequest(
    question="How many deals are in the Won stage?"
)

response_3 = await sql_analyst(test_request_3)

print("API Test Result")
print("=" * 50)
print("Success:", response_3["success"])
print("Answer:", response_3["answer"])

API Test Result
Success: True
Answer: [{'type': <SimpleComponentType.TEXT: 'text'>, 'semantic_type': None, 'metadata': None, 'text': 'There are 22 deals in the Won stage.'}]


In [20]:
test_request_4 = SQLAnalystRequest(
    question="How many deals are in the Lost stage?"
)

response_4 = await sql_analyst(test_request_4)

print("API Test Result")
print("=" * 50)
print("Success:", response_4["success"])
print("Answer:", response_4["answer"])

API Test Result
Success: True
Answer: [{'type': <SimpleComponentType.TEXT: 'text'>, 'semantic_type': None, 'metadata': None, 'text': 'There are 10 deals in the Lost stage.'}]


In [21]:
test_request_5 = SQLAnalystRequest(
    question="How many customers came from each lead source?"
)

response_5 = await sql_analyst(test_request_5)

print("API Test Result")
print("=" * 50)
print("Success:", response_5["success"])
print("Answer:", response_5["answer"])

API Test Result
Success: True
Answer: [{'type': <SimpleComponentType.TEXT: 'text'>, 'semantic_type': None, 'metadata': None, 'text': 'Here are the counts of customers coming from each lead source:\n\n- Cold Call: 11 customers\n- Email Campaign: 19 customers\n- LinkedIn: 19 customers\n- Referral: 24 customers\n- Trade Show: 17 customers\n- Website: 10 customers'}]


In [22]:
test_request_6 = SQLAnalystRequest(
    question="How many customers are handled by each sales representative?"
)

response_6 = await sql_analyst(test_request_6)

print("API Test Result")
print("=" * 50)
print("Success:", response_6["success"])
print("Answer:", response_6["answer"])

API Test Result
Success: True
Answer: [{'type': <SimpleComponentType.TEXT: 'text'>, 'semantic_type': None, 'metadata': None, 'text': 'Based on the SQL query results, here is the count of customers handled by each sales representative:\n\n- Aditi Rao: 13 customers\n- Arjun Das: 13 customers\n- Karthik Nair: 19 customers\n- Priya Shah: 20 customers\n- Rahul Menon: 19 customers\n- Sneha Iyer: 16 customers'}]


In [23]:
test_request_7 = SQLAnalystRequest(
    question="Who is the customer with the highest deal value? Show the customer name, company, and deal value."
)

response_7 = await sql_analyst(test_request_7)

print("API Test Result")
print("=" * 50)
print("Success:", response_7["success"])
print("Answer:", response_7["answer"])

API Test Result
Success: True
Answer: [{'type': <SimpleComponentType.TEXT: 'text'>, 'semantic_type': None, 'metadata': None, 'text': 'The customer with the highest deal value is Ananya Iyer from the company DeltaWorks, with a deal value of ₹5,00,000.'}]


In [24]:
test_request_8 = SQLAnalystRequest(
    question="What is the average deal value of all customers?"
)

response_8 = await sql_analyst(test_request_8)

print("API Test Result")
print("=" * 50)
print("Success:", response_8["success"])
print("Answer:", response_8["answer"])

API Test Result
Success: True
Answer: [{'type': <SimpleComponentType.TEXT: 'text'>, 'semantic_type': None, 'metadata': None, 'text': 'The average deal value of all customers is ₹248,750.00.'}]


In [25]:
test_request_9 = SQLAnalystRequest(
    question="How many deals are currently in the Negotiation stage?"
)

response_9 = await sql_analyst(test_request_9)

print("API Test Result")
print("=" * 50)
print("Success:", response_9["success"])
print("Answer:", response_9["answer"])

API Test Result
Success: True
Answer: [{'type': <SimpleComponentType.TEXT: 'text'>, 'semantic_type': None, 'metadata': None, 'text': 'There are currently 11 deals in the Negotiation stage.'}]


In [26]:
test_request_10 = SQLAnalystRequest(
    question="What is the distribution of customers across different deal stages?"
)

response_10 = await sql_analyst(test_request_10)

print("API Test Result")
print("=" * 50)
print("Success:", response_10["success"])
print("Answer:", response_10["answer"])

API Test Result
Success: True
Answer: [{'type': <SimpleComponentType.TEXT: 'text'>, 'semantic_type': None, 'metadata': None, 'text': 'The distribution of customers across different deal stages is as follows:\n\n- Lost: 10 customers\n- Negotiation: 11 customers\n- New Lead: 14 customers\n- Proposal: 15 customers\n- Qualified: 28 customers\n- Won: 22 customers\n\nThis distribution is based on the data available in the `deal_stage` column of the `customers` table.'}]


In [27]:
import pandas as pd
import sqlite3

verification_queries = {
    "Customer Count":
        "SELECT COUNT(*) AS result FROM customers",

    "Total Deal Value":
        "SELECT SUM(deal_value) AS result FROM customers",

    "Won Deals":
        "SELECT COUNT(*) AS result FROM customers WHERE deal_stage = 'Won'",

    "Lost Deals":
        "SELECT COUNT(*) AS result FROM customers WHERE deal_stage = 'Lost'",

    "Negotiation Deals":
        "SELECT COUNT(*) AS result FROM customers WHERE deal_stage = 'Negotiation'",

    "Average Deal Value":
        "SELECT AVG(deal_value) AS result FROM customers"
}

with sqlite3.connect(db_path) as conn:
    for name, query in verification_queries.items():
        result = pd.read_sql_query(query, conn)
        print(f"{name}: {result.iloc[0, 0]}")

Customer Count: 100
Total Deal Value: 24875000
Won Deals: 22
Lost Deals: 10
Negotiation Deals: 11
Average Deal Value: 248750.0


In [28]:
from fastapi.testclient import TestClient

client = TestClient(app)

http_response = client.post(
    "/cia/sql-analyst",
    json={
        "question": "How many customers are there in the CRM database?"
    }
)

print("HTTP Status Code:", http_response.status_code)
print("Response JSON:")
print(http_response.json())

c:\Users\Prave\OneDrive\Desktop\SQL-Day1\SQL-Day3\Pandas-Advanced\Week-3-Day-5\Week-4\Week-4-Day-5-Capstone\Week-5-Day-1-PandasAI\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


HTTP Status Code: 200
Response JSON:
{'success': True, 'conversation_id': '9682676e-e4c7-433d-bae1-82aa71ef4995', 'request_id': '753d94bf-ba18-4035-ae86-ef9d02ec0129', 'answer': [{'type': 'text', 'semantic_type': None, 'metadata': None, 'text': 'There are 100 customers in the CRM database.'}]}


In [29]:
e2e_response = client.post(
    "/cia/sql-analyst",
    json={
        "question": "How many deals are in the Won stage?"
    }
)

print("HTTP Status:", e2e_response.status_code)

e2e_data = e2e_response.json()

print("API Success:", e2e_data["success"])
print("Conversation ID:", e2e_data["conversation_id"])
print("Answer:", e2e_data["answer"])

HTTP Status: 200
API Success: True
Conversation ID: 7d640be7-064a-4b95-9ebe-43921187f29c
Answer: [{'type': 'text', 'semantic_type': None, 'metadata': None, 'text': 'There are 22 deals in the Won stage.'}]


In [30]:
import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path.cwd().parent.parent / "crm_database.db"

conn = sqlite3.connect(db_path)

# 1. Customers by Lead Source
lead_source_check = pd.read_sql_query("""
SELECT lead_source, COUNT(*) AS customer_count
FROM customers
GROUP BY lead_source
ORDER BY lead_source
""", conn)

# 2. Customers by Sales Representative
sales_rep_check = pd.read_sql_query("""
SELECT sales_rep, COUNT(*) AS customer_count
FROM customers
GROUP BY sales_rep
ORDER BY sales_rep
""", conn)

# 3. Customer with Highest Deal Value
highest_deal_check = pd.read_sql_query("""
SELECT customer_name, company, deal_value
FROM customers
ORDER BY deal_value DESC
LIMIT 1
""", conn)

# 4. Deal Stage Distribution
stage_distribution_check = pd.read_sql_query("""
SELECT deal_stage, COUNT(*) AS customer_count
FROM customers
GROUP BY deal_stage
ORDER BY deal_stage
""", conn)

print("1. Customers by Lead Source")
display(lead_source_check)

print("2. Customers by Sales Representative")
display(sales_rep_check)

print("3. Highest Deal Value")
display(highest_deal_check)

print("4. Deal Stage Distribution")
display(stage_distribution_check)

conn.close()

1. Customers by Lead Source


,lead_source,customer_count
0,Cold Call,11
1,Email Campaign,19
2,LinkedIn,19
3,Referral,24
4,Trade Show,17
5,Website,10


2. Customers by Sales Representative


,sales_rep,customer_count
0,Aditi Rao,13
1,Arjun Das,13
2,Karthik Nair,19
3,Priya Shah,20
4,Rahul Menon,19
5,Sneha Iyer,16


3. Highest Deal Value


,customer_name,company,deal_value
0,Ananya Iyer,DeltaWorks,500000


4. Deal Stage Distribution


,deal_stage,customer_count
0,Lost,10
1,Negotiation,11
2,New Lead,14
3,Proposal,15
4,Qualified,28
5,Won,22
